In [1]:
# 城市数据：合并 data/搜索关键词 下各城市 CSV（排除“全国”）并保留指定字段
from pathlib import Path
import re
import pandas as pd

TARGET_COLUMNS = [
    '职位名称', '薪资', '薪资类型', '薪资发放次数', '工作城市', '行政区', '商圈/街道',
    '工作地点展示', '详细地址', '经度', '纬度', '经验要求', '学历要求', '工作类型',
    '工作模式', '职位类别', '公司名称', '公司规模', '公司性质', '融资阶段', '行业',
    '发布时间', '首次发布时间', '发布日期文本', '是否新职位', '招聘人数', 'HR状态',
    'HR活跃标签', '职位标签汇总', '技能标签', '福利标签', '福利明细', '工作时间',
    '报告项/保障项', '职位描述', '职位亮点', '职位摘要', '认证/守护信息',
]

# 手动填写要处理的搜索关键词，必须与 data 下的目录名和 CSV 文件名中的关键词一致。
# 例如：data/数据分析/北京_数据分析.csv -> SEARCH_KEYWORD = '数据分析'
SEARCH_KEYWORD = '数据开发'

GENERIC_SOURCE_FILE_RE = re.compile(r'(?P<city>.+?)_(?P<keyword>.+?)[.]csv$', re.IGNORECASE)
NATIONAL_RE = re.compile(r'全国|全國|national|nationwide', re.IGNORECASE)

EXTRA_COLUMN_PATTERNS = {
    '职位名称': r'(positionname|jobname|jobtitle|岗位名称|岗位标题|职位标题)',
    '薪资': r'(salary60|salaryreal|salary|薪酬|工资|薪水)',
    '薪资类型': r'(salarytype|薪酬类型|工资类型)',
    '薪资发放次数': r'(salarycount|发薪次数|发放次数)',
    '工作城市': r'(workcity|cityname|城市名称|城市)',
    '行政区': r'(citydistrict|district|行政区|区县)',
    '商圈/街道': r'(streetname|tradingarea|商圈|街道)',
    '工作地点展示': r'(displayaddress|worklocationtext|工作地点|地点展示)',
    '详细地址': r'(workaddress|detailaddress|详细地址|地址)',
    '经度': r'(longitude|lng|经度)',
    '纬度': r'(latitude|lat|纬度)',
    '经验要求': r'(workingexp|experience|工作经验|经验)',
    '学历要求': r'(education|学历)',
    '工作类型': r'(worktype|职位类型|岗位类型)',
    '工作模式': r'(workmode|办公模式|工作模式)',
    '职位类别': r'(subjobtypelevelname|jobcategory|职位类别|岗位类别|职能)',
    '公司名称': r'(companyname|企业名称|公司)',
    '公司规模': r'(companysize|公司人数|企业规模)',
    '公司性质': r'(propertyname|property|companynature|企业性质)',
    '融资阶段': r'(financingstage|融资)',
    '行业': r'(industryname|industry|行业)',
    '发布时间': r'(publishtime|positionpublishtime|职位发布时间)',
    '首次发布时间': r'(firstpublishtime|首次发布)',
    '发布日期文本': r'(positionupdatetimetext|发布时间文本|发布日期文本)',
    '是否新职位': r'(isnewposition|newposition|新职位)',
    '招聘人数': r'(recruitnumber|招聘数量|招聘人数)',
    'HR状态': r'(hrstate|hronlinestate|hr状态|hr在线状态)',
    'HR活跃标签': r'(activitylevel|hractivity|hr活跃|活跃标签)',
    '职位标签汇总': r'(jobtags|showskilltags|职位标签|岗位标签)',
    '技能标签': r'(skilllabel|jobskilltags|技能|技能标签)',
    '福利标签': r'(welfarelabel|welfaretags|福利标签)',
    '福利明细': r'(welfareitems|福利明细)',
    '工作时间': r'(worktimeitems|工作时间)',
    '报告项/保障项': r'(reportitems|promiseguarantee|报告项|保障项)',
    '职位描述': r'(description|jobdescription|职位描述|岗位描述)',
    '职位亮点': r'(positionhighlight|descriptionhighlight|职位亮点|岗位亮点)',
    '职位摘要': r'(jobsummary|职位摘要|岗位摘要)',
    '认证/守护信息': r'(verifythetruth|认证守护信息|认证信息|守护信息)',
}

def normalize_name(name):
    text = str(name).replace('\ufeff', '').lower()
    for ch in [' ', '\t', '\r', '\n', '\u3000', '_', '-', '.', '/', '\\', '|', ':', '：', ';', '；', ',', '，', '(', ')', '（', '）', '[', ']', '【', '】']:
        text = text.replace(ch, '')
    return text

def build_column_regex(target):
    exact = re.escape(normalize_name(target))
    extra = EXTRA_COLUMN_PATTERNS.get(target)
    pattern = f'^(?:{exact})$'
    if extra:
        pattern = f'{pattern}|(?:{extra})'
    return re.compile(pattern, re.IGNORECASE)

COLUMN_REGEX = {column: build_column_regex(column) for column in TARGET_COLUMNS}

def keyword_dirs_under(root):
    if not root.exists():
        return []
    folders = []
    scan_dirs = [root]
    scan_dirs.extend(path for path in root.iterdir() if path.is_dir() and path.name not in {'cleaned', '清洗后的数据'})
    for folder in scan_dirs:
        for path in folder.glob('*.csv'):
            match = GENERIC_SOURCE_FILE_RE.match(path.name)
            if match and normalize_name(match.group('keyword')) == normalize_name(folder.name):
                folders.append(folder)
                break
    return sorted(set(folder.resolve() for folder in folders))

def has_source_csv(root):
    return bool(keyword_dirs_under(root))

def locate_data_root():
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([
            base,
            base / 'data',
            base / '爬虫' / 'data',
        ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        keyword_dirs = keyword_dirs_under(candidate)
        if keyword_dirs:
            if candidate in keyword_dirs:
                return candidate.parent
            return candidate
    raise FileNotFoundError('没有找到 data/搜索关键词/城市_关键词.csv 格式的数据文件，请确认 notebook 在项目目录、爬虫目录或 data 目录下运行。')

DATA_ROOT = locate_data_root()

def choose_keyword_dir():
    keyword = str(SEARCH_KEYWORD).strip()
    if not keyword:
        raise ValueError("请在第一个单元格顶部手动填写 SEARCH_KEYWORD，例如：SEARCH_KEYWORD = '数据分析'")
    folder = DATA_ROOT / keyword
    if not folder.exists():
        raise FileNotFoundError(f'指定的搜索关键词目录不存在: {folder}')
    return folder.resolve()

KEYWORD_DIR = choose_keyword_dir()
ACTIVE_KEYWORD = KEYWORD_DIR.name
OUTPUT_DIR = DATA_ROOT.parent / '数据清洗' / '合并并筛选' / '清洗后的数据'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def list_source_files(include_national):
    files = []
    for path in sorted(KEYWORD_DIR.glob('*.csv')):
        match = GENERIC_SOURCE_FILE_RE.match(path.name)
        if not match:
            continue
        city = match.group('city')
        keyword = match.group('keyword')
        if normalize_name(keyword) != normalize_name(ACTIVE_KEYWORD):
            continue
        is_national = NATIONAL_RE.search(city) is not None
        if include_national == is_national:
            files.append((path, city))
    return files

def read_csv_with_fallback(path):
    last_error = None
    for encoding in ['utf-8-sig', 'utf-8', 'gb18030']:
        try:
            return pd.read_csv(path, dtype=str, encoding=encoding, keep_default_na=False)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

def match_column(columns, target, used_columns):
    normalized_target = normalize_name(target)
    for column in columns:
        if column in used_columns:
            continue
        if normalize_name(column) == normalized_target:
            return column
    regex = COLUMN_REGEX[target]
    for column in columns:
        if column in used_columns:
            continue
        if regex.search(normalize_name(column)):
            return column
    return None

def clean_series(series):
    return series.astype(str).str.replace('\r\n', '\n', regex=False).str.strip()

def select_target_columns(df):
    cleaned = pd.DataFrame(index=df.index)
    used_columns = set()
    missing = []
    for target in TARGET_COLUMNS:
        source = match_column(df.columns, target, used_columns)
        if source is None:
            cleaned[target] = ''
            missing.append(target)
        else:
            cleaned[target] = clean_series(df[source])
            used_columns.add(source)
    return cleaned[TARGET_COLUMNS], missing

def print_missing_report(missing_report):
    if not missing_report:
        print('所有目标字段都已匹配。')
        return
    print('以下文件缺少部分目标字段，已用空值补齐：')
    for filename, missing_columns in missing_report.items():
        missing_text = ', '.join(missing_columns)
        print(f'- {filename}: {missing_text}')

city_files = list_source_files(include_national=False)
if not city_files:
    raise FileNotFoundError(f'在 {DATA_ROOT} 下没有找到城市 CSV 文件。')

city_frames = []
city_missing_report = {}
for path, city in city_files:
    source_df = read_csv_with_fallback(path)
    cleaned_df, missing_columns = select_target_columns(source_df)
    city_frames.append(cleaned_df)
    if missing_columns:
        city_missing_report[path.name] = missing_columns

city_df = pd.concat(city_frames, ignore_index=True)
CITY_OUTPUT = OUTPUT_DIR / f'自定义范围_{ACTIVE_KEYWORD}_清洗.csv'
city_df.to_csv(CITY_OUTPUT, index=False, encoding='utf-8-sig')

print(f'数据根目录: {DATA_ROOT}')
print(f'搜索关键词目录: {KEYWORD_DIR}')
print(f'当前搜索关键词: {ACTIVE_KEYWORD}')
print(f'城市文件数量: {len(city_files)}')
print(f'城市合并行数: {len(city_df):,}')
print(f'保留字段数: {len(TARGET_COLUMNS)}')
print(f'输出文件: {CITY_OUTPUT}')
print_missing_report(city_missing_report)
try:
    display(city_df.head())
except NameError:
    print(city_df.head().to_string(index=False))


数据根目录: D:\桌面\实训项目\data
搜索关键词目录: D:\桌面\实训项目\data\数据开发
当前搜索关键词: 数据开发
城市文件数量: 24
城市合并行数: 1,831
保留字段数: 38
输出文件: D:\桌面\实训项目\数据清洗\合并并筛选\清洗后的数据\自定义范围_数据开发_清洗.csv
所有目标字段都已匹配。


,职位名称,薪资,薪资类型,薪资发放次数,工作城市,行政区,商圈/街道,工作地点展示,详细地址,经度,...,职位标签汇总,技能标签,福利标签,福利明细,工作时间,报告项/保障项,职位描述,职位亮点,职位摘要,认证/守护信息
0,银行开发工程师 / 数据分析师（金融方向）,1-1.5万,1,,上海,浦东,陆家嘴,上海 浦东 陆家嘴,,,...,基金数据分析 | SQL | C++ | MySQL | 本科 | 银行,基金数据分析 | SQL | C++ | MySQL | 本科 | 银行,,,,,岗位职责<div> \n1. 负责银行内部业务报表的需求沟通、设计、开发与测试工作。 \n2...,,,实名核验 ・ 资质核验
1,数据开发（微软云经验）,2.2-2.5万,1,,上海,静安,石门二路,上海 静安 石门二路,,,...,Azure | 本科 | 5-10年,Azure | 本科 | 5-10年,,,,,良好的沟通和演示技巧，能够清晰沟通并向他人解释技术解决方案。\n一个快速学习者，出色的队友。...,,,
2,数据开发,8000-13000元,1,,上海,崇明,南京东路,上海 崇明 南京东路,,,...,专人带教 | 五险一金 | 大牛带队,专人带教 | 五险一金,,,,,岗位职责\n通过线上聊天渠道解答用户常规疑问，做好咨询内容登记。\n整理用户高频问题汇总上交...,,,
3,服务零售-数据开发工程师,面议,1,,上海,杨浦,大桥,上海 杨浦 大桥,,,...,本科 | 3-5年 | 快速提升 | 互联网100强 | 最佳雇主,本科 | 3-5年 | 快速提升,,,,,基础研发平台是美团的核心技术平台，立足于“零售+科技”的战略定位，通过打造人工智能、大数据、...,,,
4,Java/ Web/ C开发/ 测试/算法/数据开发,1.4-2.8万·16薪,1,,上海,浦东,金桥,上海 浦东 金桥,,,...,JavaScript | Python | C++ | Spring | Mybatis |...,JavaScript | Python | C++ | Spring | Mybatis |...,,,,,岗位要求】\n1、 本科以上毕业；\n2、 具备Java/python/C++/C任意一种开...,,,


In [2]:
# 全国数据：单独清洗“全国”CSV，并保留同一组指定字段
if 'TARGET_COLUMNS' not in globals():
    raise RuntimeError('请先运行上一个“城市数据”单元格，再运行本单元格。')

national_files = list_source_files(include_national=True)
if not national_files:
    raise FileNotFoundError(f'在 {DATA_ROOT} 下没有找到“全国”CSV 文件。')

national_frames = []
national_missing_report = {}
for path, city in national_files:
    source_df = read_csv_with_fallback(path)
    cleaned_df, missing_columns = select_target_columns(source_df)
    national_frames.append(cleaned_df)
    if missing_columns:
        national_missing_report[path.name] = missing_columns

national_df = pd.concat(national_frames, ignore_index=True)
NATIONAL_OUTPUT = OUTPUT_DIR / f'全国范围_{ACTIVE_KEYWORD}_清洗.csv'
national_df.to_csv(NATIONAL_OUTPUT, index=False, encoding='utf-8-sig')

print(f'数据根目录: {DATA_ROOT}')
print(f'搜索关键词目录: {KEYWORD_DIR}')
print(f'当前搜索关键词: {ACTIVE_KEYWORD}')
print(f'全国文件数量: {len(national_files)}')
print(f'全国数据行数: {len(national_df):,}')
print(f'保留字段数: {len(TARGET_COLUMNS)}')
print(f'输出文件: {NATIONAL_OUTPUT}')
print_missing_report(national_missing_report)
try:
    display(national_df.head())
except NameError:
    print(national_df.head().to_string(index=False))


数据根目录: D:\桌面\实训项目\data
搜索关键词目录: D:\桌面\实训项目\data\数据开发
当前搜索关键词: 数据开发
全国文件数量: 1
全国数据行数: 1,000
保留字段数: 38
输出文件: D:\桌面\实训项目\数据清洗\合并并筛选\清洗后的数据\全国范围_数据开发_清洗.csv
所有目标字段都已匹配。


,职位名称,薪资,薪资类型,薪资发放次数,工作城市,行政区,商圈/街道,工作地点展示,详细地址,经度,...,职位标签汇总,技能标签,福利标签,福利明细,工作时间,报告项/保障项,职位描述,职位亮点,职位摘要,认证/守护信息
0,数据开发部时空数据平台项目经理,1.2-1.5万,1,,株洲,石峰区,,株洲 石峰区,,,...,本科 | 1-3年 | 数据资源运营 | 数据产品交易 | 云计算,本科 | 1-3年 | 数据资源运营 | 数据产品交易 | 云计算,,,,,岗位职责：<div> 1.负责指导开发团队完成时空数据交易平台和底层功能模块、硬件系统的设计...,,,
1,数据开发工程师,9000-12000元,1,,青岛,市南,金门路,青岛 市南 金门路,,,...,Oracle | MySQL | SQL Server | 本科 | 3-5年 | 优选雇主,Oracle | MySQL | SQL Server | 本科 | 3-5年,,,,,岗位职责:\n1. 负责 MS-SQL 数据库体系建设与优化：\n• 根据业务需求，参与设计...,,,
2,数据开发工程师（反洗钱项目，可出差）,1.2-1.6万,1,,南京,建邺,沙洲,南京 建邺 沙洲,,,...,ETL | Sql | 银行项目 | 反洗钱 | 本科 | 3-5年,ETL | Sql | 银行项目 | 反洗钱 | 本科 | 3-5年,,,,,- 统招本科，双证齐全，计算机相关专业，熟练掌握ETL开发工具（如DataStage、Inf...,,,
3,数据开发岗（AI方向）,面议,1,,长沙,岳麓,观沙岭,长沙 岳麓 观沙岭,,,...,AGENT | 深度学习 | Java | Python | 本科 | 5-10年,AGENT | 深度学习 | Java | Python | 本科 | 5-10年,,,,,"【岗位职责】\n1.使用大型语言模型（如DeepSeek,Qwen等）构建和部署自助AI A...",,,实名核验 ・ 资质核验
4,医疗数据开发工程师,1-1.7万·13薪,1,,北京,海淀,,北京 海淀,,,...,数据挖掘 | 数据库开发 | 数据中台开发 | Python | 本科 | 1-3年 | 发...,数据挖掘 | 数据库开发 | 数据中台开发 | Python | 本科 | 1-3年 | 发...,,,,,<div> 岗位职责</div><div> <ol> \n <li>对接合作医院，...,,,
